In [ ]:
from pathlib import Path
import json, gzip, hashlib, os, sys
from collections import Counter, defaultdict
from importlib.metadata import version, PackageNotFoundError
root = Path.cwd()
data = root / 'datasets/dataset'
out = root / 'runs/dinov2_material_stage1'
out.mkdir(parents=True, exist_ok=True)
def readj(p): return json.loads(Path(p).read_text())
labels = readj(data/'labels/real_train.json')
split = readj(root/'baseline_v0/split_v0.json')
print('Workspace:',root)
for pkg in ['torch','torchvision','transformers','timm','huggingface_hub','safetensors','scikit-learn']:
    try: print(pkg,version(pkg))
    except PackageNotFoundError: print(pkg,'absent')
ims={i['id']:i for i in labels['images']}
print('annotations',len(labels['annotations']), 'sample', {k:v for k,v in labels['annotations'][0].items() if k!='segmentation'})
print('image',next(iter(ims.values())))
for part in ['train','val']:
    rows=[a['attributes'] for a in labels['annotations'] if Path(ims[a['image_id']]['file_name']).parent.name in split[part]['image_ids']]
    print(part,Counter(x['material'] for x in rows), Counter(dict((x['object_id'],x['material']) for x in rows).values()))
print('notebooks', [str(p.relative_to(root)) for p in root.rglob('*.ipynb')][:70]); print('score',[str(p.relative_to(root)) for p in root.rglob('score.py')]);print('cache',[str(p.relative_to(root)) for p in (root/'runs').glob('*')]); print('weights', [str(p) for base in [root,Path.home()/'.cache'] if base.exists() for p in base.rglob('*dino*')][:60])

In [ ]:
prior=root/'runs/material_dinov2_v1_20261122'
for d in [prior,root/'runs/material_final_m_20261121_7f35d05_oof4']:
    print(d.name,[(str(p.relative_to(d)),p.stat().st_size) for p in d.rglob('*') if p.is_file() and 'vendor' not in p.parts][:120])
print('official notebook', [(c['cell_type'], ''.join(c['source']).strip().split('\n')[0][:150]) for c in readj(root/'datasets/baseline.ipynb')['cells'] if c['source']]); print('score functions',[l for l in (root/'datasets/score.py').read_text().splitlines() if l.startswith('def ')])
print('weights',[(str(p),p.stat().st_size) for p in prior.rglob('*') if p.is_file() and p.stat().st_size>20_000_000]); print('done')

In [ ]:
old=root/'runs/material_final_m_20261121_7f35d05_oof4'
protocol=readj(old/'protocol.json'); preds=readj(old/'oof_predictions.json'); print('protocol',str(protocol)[:4100]);print('pred structure',[(k,type(v).__name__,str(v)[:1100]) for k,v in preds.items()] if isinstance(preds,dict) else preds[0]); print('source',readj(prior/'weights_source.json')); print('match and metric', (root/'datasets/score.py').read_text().split('def _match')[1].split('def score')[0][:5400]);print('vendor',list((prior/'vendor').iterdir()))

In [ ]:
historical=readj(root/'runs/take_probability_20261121_e4374505/train.json'); previous=readj(old/'results.json')
print('old fold keys',[(k,type(v).__name__,str(v)[:220]) for k,v in historical.items()]);print('results',[(k,str(v)[:200]) for k,v in previous.items()]); vend=next((prior/'vendor').iterdir()); print('preprocessing', (vend/'dinov2/data/transforms.py').read_text().split('def make_classification_eval_transform')[1][:1300]); print('baseline split code', ''.join(readj(root/'datasets/baseline.ipynb')['cells'][4]['source'])[:2600]); print('val caches', [(str(p),p.stat().st_size) for p in (root/'runs').rglob('*val*.json')][:35])

In [ ]:
from pathlib import Path
import subprocess, sys
expdir=Path('/home/jovyan/work/runs/dinov2_material_stage1')
expdir.mkdir(parents=True,exist_ok=True)
script=expdir/"extract_embeddings.py"
script.write_text("from pathlib import Path\nimport hashlib, json, math, sys\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torchvision import transforms as T\n\nROOT = Path('/home/jovyan/work')\nOUT = Path(__file__).resolve().parent\nDATA = ROOT/'datasets/dataset'\nPRIOR = ROOT/'runs/material_dinov2_v1_20261122'\nOLD = ROOT/'runs/material_final_m_20261121_7f35d05_oof4'\nread = lambda p: json.loads(p.read_text())\nsave = lambda p, x: p.write_text(json.dumps(x, indent=2, ensure_ascii=False))\n\ndef main():\n    params = read(PRIOR/'weights_source.json')\n    weights = PRIOR/'dinov2_vits14_pretrain.pth'\n    digest = hashlib.sha256(weights.read_bytes()).hexdigest()\n    assert digest == params[weights.name]['sha256'] == 'b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d8d00217f29aef60cd9'\n    torch.set_num_threads(2)\n    source = PRIOR/'vendor/dinov2-7764ea0f912e53c92e82eb78a2a1631e92725fc8'\n    sys.path.insert(0,str(source))\n    from dinov2.hub.backbones import dinov2_vits14\n    encoder = dinov2_vits14(pretrained=False)\n    encoder.load_state_dict(torch.load(weights,map_location='cpu',weights_only=True),strict=True)\n    encoder.requires_grad_(False).eval()\n    assert encoder.embed_dim == 384 and not any(p.requires_grad for p in encoder.parameters())\n    prep = T.Compose([T.Resize(256,interpolation=T.InterpolationMode.BICUBIC),T.CenterCrop(224),T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])\n    labels=read(DATA/'labels/real_train.json'); images={im['id']:im for im in labels['images']}\n    paths={Path(im['file_name']).parent.name:DATA/im['file_name'] for im in images.values()}\n    rows=[]\n    for a in labels['annotations']:\n        frame=Path(images[a['image_id']]['file_name']).parent.name\n        x,y,w,h=a['bbox']; attr=a['attributes']\n        rows.append(dict(kind='gt',frame=frame,box=[x,y,x+w,y+h],object_id=attr['object_id'],material=attr['material'],ann=a['id']))\n    baseline=read(OLD/'oof_predictions.json')['A']\n    for frame,dets in baseline.items():\n        for j,det in enumerate(dets): rows.append(dict(kind='det',frame=frame,index=j,box=det['box']))\n    batches=[]; batch=[]\n    with torch.inference_mode():\n        for i,row in enumerate(rows):\n            path=paths[row['frame']]\n            if not path.exists(): path=path.with_suffix('.jpg')\n            with Image.open(path) as im:\n                x1,y1,x2,y2=row['box']; box=(max(0,math.floor(x1)),max(0,math.floor(y1)),min(im.width,math.ceil(x2)),min(im.height,math.ceil(y2)))\n                assert box[2]>box[0] and box[3]>box[1]\n                batch.append(prep(im.convert('RGB').crop(box)))\n            if len(batch)==8 or i+1==len(rows):\n                batches.append(encoder(torch.stack(batch)).cpu().numpy()); batch.clear()\n    X=np.concatenate(batches); assert X.shape==(len(rows),384) and np.isfinite(X).all()\n    np.save(OUT/'embeddings.npy',X); save(OUT/'crops.json',rows); save(OUT/'encoder.json',dict(source=params,embedding='CLS after final layer normalization',transform='RGB, bbox floor/ceil clipped, Resize(256,bicubic), CenterCrop(224), ImageNet normalization',torch=torch.__version__))\n    print('SAVED',X.shape,'sha256',digest,flush=True)\n\nif __name__=='__main__': main()\n",encoding='utf-8')
job=subprocess.run([sys.executable,str(script)],capture_output=True,text=True)
print(job.stdout[-5500:],job.stderr[-3500:])
job.check_returncode()

In [ ]:
from pathlib import Path
import subprocess, sys
expdir=Path('/home/jovyan/work/runs/dinov2_material_stage1')
expdir.mkdir(parents=True,exist_ok=True)
script=expdir/"group_oof.py"
script.write_text("from pathlib import Path\nfrom collections import defaultdict, Counter\nimport copy, json, sys\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score\nfrom threadpoolctl import threadpool_limits\nfrom extract_embeddings import ROOT, OUT, DATA, OLD, read, save\nsys.path.insert(0,str(ROOT/'datasets'))\nimport score as official\n\nCLASSES=[\"metal\",\"wood\",\"plastic\",\"glass\",\"paper\",\"mineral\"]\nassert set(CLASSES)==set(official.MATERIALS)\nGRID=[.0001,.001,.01,.1,1.0]\n\ndef train(X,y,C):\n    return LogisticRegression(C=C, max_iter=2000, class_weight='balanced', solver='lbfgs',tol=1e-6).fit(X,y)\n\ndef tune(X,y,g):\n    # Split the distinct physical objects within each material. Singletons stay in inner-fit.\n    folds=[[] for _ in range(3)]; rng=np.random.default_rng(2718)\n    for label in sorted(set(y)):\n        ids=np.unique(g[y==label]);rng.shuffle(ids)\n        if len(ids)>1:\n            for j,obj in enumerate(ids): folds[j%3].append(str(obj))\n    scores={}\n    for c in GRID:\n        yt=[];yp=[]\n        for held in folds:\n            test=np.isin(g,held)\n            if not test.any():continue\n            assert set(g[test]).isdisjoint(g[~test]); assert set(y[~test])==set(y)\n            pred=train(X[~test],y[~test],c).predict(X[test]);yt.extend(y[test]);yp.extend(pred)\n        scores[c]=float(recall_score(yt,yp,labels=sorted(set(yt)),average='macro',zero_division=0))\n    best=max(GRID,key=lambda c:scores[c]);return best,dict(C=best,scores=scores,inner_validation_objects=folds)\n\ndef metrics(yt,yp):\n    return dict(n=len(yt),accuracy=float(accuracy_score(yt,yp)),macro_f1=float(f1_score(yt,yp,labels=CLASSES,average='macro',zero_division=0)),confusion=confusion_matrix(yt,yp,labels=CLASSES+['miss']).tolist()[:len(CLASSES)],recall={m:float(np.mean(np.array(yp)[np.array(yt)==m]==m)) for m in CLASSES if m in yt})\n\ndef head(name,model,objects,c):\n    np.savez(OUT/name,weight=model.coef_,bias=model.intercept_,classes=model.classes_,fit_objects=np.unique(objects),C=c)\n    with np.load(OUT/name) as z:\n        assert np.array_equal(model.predict(X),z['classes'][(X@z['weight'].T+z['bias']).argmax(axis=1)])\n\ndef run():\n    global X\n    rows=read(OUT/'crops.json');X=np.load(OUT/'embeddings.npy'); split=read(ROOT/'baseline_v0/split_v0.json')\n    anns=read(DATA/'labels/real_train.json'); ims={i['id']:Path(i['file_name']).parent.name for i in anns['images']}\n    truth={f:[] for f in split['train']['image_ids']}; validation=set(split['val']['image_ids'])\n    for a in anns['annotations']:\n        f=ims[a['image_id']]; x,y,w,h=a['bbox']; v=a['attributes']\n        if f in truth:truth[f].append(dict(box=[x,y,x+w,y+h],form=v['form'],material=v['material'],volume=v['volume_ml'],take=v['take']))\n    for f in truth: assert len(truth[f])<=1\n    base=read(OLD/'oof_predictions.json')['A']; assert set(base)==set(truth)\n    candidate=copy.deepcopy(base); gtrows=[i for i,r in enumerate(rows) if r['kind']=='gt']; tidx=np.array([i for i in gtrows if rows[i]['frame'] in truth]); vidx=np.array([i for i in gtrows if rows[i]['frame'] in validation]); assert len(tidx)+len(vidx)==len(gtrows)\n    fn=np.array([rows[i]['frame'] for i in tidx]); y=np.array([rows[i]['material'] for i in tidx]); g=np.array([rows[i]['object_id'] for i in tidx]); frame_obj=dict(zip(fn,g));detidx={(r['frame'],r['index']):i for i,r in enumerate(rows) if r['kind']=='det'}\n    heldfolds=read(ROOT/'runs/take_probability_20261121_e4374505/train.json')['folds']; gtout={}; foldinfo=[]; done=set()\n    for d in heldfolds:\n        fold=d['fold']; held=d['held_object_ids']; valid=np.isin(g,held); fit=~valid\n        assert set(g[valid])==set(held) and set(g[valid]).isdisjoint(done) and set(y[fit])==set(CLASSES);done.update(held)\n        C,info=tune(X[tidx[fit]],y[fit],g[fit]); model=train(X[tidx[fit]],y[fit],C)\n        gtout.update(zip(fn[valid],model.predict(X[tidx[valid]])))\n        frames=set(fn[valid]); frames.update(f for f in truth if not truth[f] and fold==1)\n        for f in frames:\n            for j,p in enumerate(candidate[f]):p['material']=str(model.predict(X[[detidx[f,j]]])[0])\n        targets={f:truth[f] for f in frames}; info.update(fold=fold,held=held,fit_objects=sorted(set(g[fit])),support=dict(Counter(y[valid])),M_baseline=official.components(targets,{f:base[f] for f in frames})['M'],M_dino=official.components(targets,{f:candidate[f] for f in frames})['M']);foldinfo.append(info)\n    assert done==set(g) and set(gtout)==set(fn) and set(g).isdisjoint({rows[i]['object_id'] for i in vidx})\n    records=[]\n    for f,t,obj in zip(fn,y,g):\n        matched=lambda p:official._match(truth[f],p[f])[0]\n        b=matched(base); c=matched(candidate)\n        records.append(dict(frame=str(f),object_id=str(obj),truth=str(t),GT_DINO=str(gtout[f]),det_DINO=c['material'] if c else 'miss',det_RF=b['material'] if b else 'miss'))\n    summaries={k:metrics(list(y),[r[k] for r in records]) for k in ['GT_DINO','det_DINO','det_RF']}\n    for key,p in [('det_DINO',candidate),('det_RF',base)]:summaries[key]['M']=official.components(truth,p)['M']\n    assert abs(summaries['det_RF']['M']-read(OLD/'results.json')['metrics']['A']['M'])<1e-10\n    by_object={}\n    for obj in sorted(set(g)):\n        sub=[r for r in records if r['object_id']==obj]; by_object[obj]={k:sum(r[k]==r['truth'] for r in sub)/len(sub) for k in summaries};by_object[obj]['n']=len(sub)\n    C,train_info=tune(X[tidx],y,g); model=train(X[tidx],y,C);head('head_train32.npz',model,g,C)\n    yv=[rows[i]['material'] for i in vidx]; pv=list(model.predict(X[vidx])); hv=dict(n=len(yv),accuracy=float(accuracy_score(yv,pv)),support=dict(Counter(yv)),recall={m:np.mean(np.array(pv)[np.array(yv)==m]==m) for m in sorted(set(yv))},macro_f1_present=float(f1_score(yv,pv,labels=sorted(set(yv)),average='macro',zero_division=0)))\n    all_y=np.array([rows[i]['material'] for i in gtrows]); all_g=np.array([rows[i]['object_id'] for i in gtrows]); finalC,final_info=tune(X[gtrows],all_y,all_g); final=train(X[gtrows],all_y,finalC);head('head_all40.npz',final,all_g,finalC)\n    result=dict(classes=CLASSES,confusion_columns=CLASSES+['miss'],folds=foldinfo,OOF=summaries,objects=by_object,heldout_GT=hv,train32=train_info,final40=final_info)\n    save(OUT/'results.json',result);save(OUT/'oof_rows.json',records);save(OUT/'heldout_GT_rows.json',[dict(frame=rows[i]['frame'],object_id=rows[i]['object_id'],truth=t,predicted=p) for i,t,p in zip(vidx,yv,pv)])\n    print('classes',CLASSES,'OOF',summaries,'folds',[(i['M_baseline'],i['M_dino']) for i in foldinfo], 'heldout',hv,flush=True)\n\nif __name__=='__main__':\n    with threadpool_limits(limits=1): run()\n",encoding='utf-8')
job=subprocess.run([sys.executable,str(script)],capture_output=True,text=True)
print(job.stdout[-5500:],job.stderr[-3500:])
job.check_returncode()

In [ ]:
print('captures',[(str(p.relative_to(root)),p.stat().st_size) for p in root.rglob('capture*.json') if 'vendor' not in p.parts]); print('diagnostics',[(str(p.relative_to(root)),p.stat().st_size) for p in (root/'runs/diagnostic_v2_20261121').glob('*')]); print('datasets',[(p.name,p.stat().st_size) for p in (root/'runs/take_probability_20261121_e4374505').glob('*')]); print('artifacts', sorted(p.name for p in out.iterdir()))

In [ ]:
vcap=readj(root/'runs/diagnostic_v2_20261121/capture.json');print([(k,type(v).__name__,str(v)[:280]) for k,v in vcap.items()]);print('verified',[(k,len(v),list(v)[:6]) for k,v in vcap.items() if isinstance(v,dict)]);r=readj(out/'results.json'); diff=[v['det_DINO']-v['det_RF'] for v in r['objects'].values()];print('objects + = -',sum(x>0 for x in diff),sum(x==0 for x in diff),sum(x<0 for x in diff))

In [ ]:
from pathlib import Path
import subprocess, sys
expdir=Path('/home/jovyan/work/runs/dinov2_material_stage1')
expdir.mkdir(parents=True,exist_ok=True)
script=expdir/"material_classifier.py"
script.write_text("from pathlib import Path\nimport copy, hashlib, json, math, sys\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torchvision import transforms as T\nfrom extract_embeddings import DATA, ROOT, OUT, PRIOR, read, save\nfrom group_oof import official, CLASSES\n\nclass MaterialClassifier:\n    def __init__(self, head_path=OUT/'head_all40.npz'):\n        weight_path=PRIOR/'dinov2_vits14_pretrain.pth'\n        assert hashlib.sha256(weight_path.read_bytes()).hexdigest()==read(OUT/'encoder.json')['source'][weight_path.name]['sha256']\n        sys.path.insert(0,str(PRIOR/'vendor/dinov2-7764ea0f912e53c92e82eb78a2a1631e92725fc8'))\n        from dinov2.hub.backbones import dinov2_vits14\n        self.encoder=dinov2_vits14(pretrained=False)\n        self.encoder.load_state_dict(torch.load(weight_path,weights_only=True,map_location='cpu'),strict=True)\n        self.encoder.requires_grad_(False).eval()\n        self.prep=T.Compose([T.Resize(256,interpolation=T.InterpolationMode.BICUBIC),T.CenterCrop(224),T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])\n        with np.load(head_path) as h:\n            self.classes,self.weight,self.bias=h['classes'],h['weight'],h['bias']\n            self.fit_objects=set(h['fit_objects'])\n    @torch.inference_mode()\n    def embed(self,rgb_image,box):\n        x,y,right,bottom=box\n        limits=(max(0,math.floor(x)),max(0,math.floor(y)),min(rgb_image.width,math.ceil(right)),min(rgb_image.height,math.ceil(bottom)))\n        crop=rgb_image.convert('RGB').crop(limits)\n        return self.encoder(self.prep(crop).unsqueeze(0)).numpy()[0]\n    def classify(self,rgb_image,box):\n        return str(self.classes[(self.weight@self.embed(rgb_image,box)+self.bias).argmax()])\n\ndef holdout():\n    torch.set_num_threads(2)\n    capture=read(ROOT/'runs/diagnostic_v2_20261121/capture.json'); split=read(ROOT/'baseline_v0/split_v0.json')\n    truth=capture['truth'];base=capture['predictions']['material_simple_v1'];cand=copy.deepcopy(base)\n    assert set(truth)==set(base)==set(split['val']['image_ids']); model=MaterialClassifier(OUT/'head_train32.npz')\n    assert model.fit_objects.isdisjoint({g['object_id'] for objs in truth.values() for g in objs}); assert len(model.fit_objects)==32\n    labs=read(DATA/'labels/real_train.json'); paths={Path(i['file_name']).parent.name:DATA/i['file_name'] for i in labs['images']}\n    anns={i['id']:Path(i['file_name']).parent.name for i in labs['images']}\n    verified=set()\n    for a in labs['annotations']:\n        f=anns[a['image_id']]\n        if f in truth:\n            g=truth[f][0]; attr=a['attributes']; x,y,w,h=a['bbox']; assert (attr['object_id'],attr['material'])==(g['object_id'],g['material']) and np.allclose(g['box'],[x,y,x+w,y+h]); verified.add(f)\n    assert verified==set(truth)\n    embs=[]; indices=[];rows=[]\n    for f in sorted(cand):\n        path=paths[f]\n        if not path.exists():path=path.with_suffix('.jpg')\n        with Image.open(path) as im:\n            for j,p in enumerate(cand[f]):\n                z=model.embed(im,p['box']); embs.append(z); indices.append([f,j]);p['material']=str(model.classes[(model.weight@z+model.bias).argmax()])\n        for g,b,c in zip(truth[f],official._match(truth[f],base[f]),official._match(truth[f],cand[f])):\n            rows.append(dict(frame=f,object_id=g['object_id'],truth=g['material'],RF=b['material'] if b else 'miss',DINO=c['material'] if c else 'miss'))\n    result={}\n    for name,p in [('RF',base),('DINO',cand)]:\n        hits=[r[name]==r['truth'] for r in rows]; recall={m:sum(r[name]==m for r in rows if r['truth']==m)/sum(r['truth']==m for r in rows) for m in sorted({r['truth'] for r in rows})}\n        result[name]=dict(M=official.components(truth,p)['M'],accuracy=float(np.mean(hits)),recall=recall)\n    assert abs(result['RF']['M']-capture['scores']['material_simple_v1']['M'])<1e-10\n    np.save(OUT/'heldout_detector_embeddings.npy',np.stack(embs));save(OUT/'heldout_detector_indices.json',indices);save(OUT/'heldout_detector.json',dict(metrics=result,rows=rows));print('Independent holdout',len(rows),len(indices),result)\n\nif __name__=='__main__':holdout()\n",encoding='utf-8')
job=subprocess.run([sys.executable,str(script)],capture_output=True,text=True)
print(job.stdout[-5500:],job.stderr[-3500:])
job.check_returncode()

In [ ]:
from importlib.metadata import version
import numpy as np
import subprocess, sys
names=['results.json','oof_rows.json','head_train32.npz','head_all40.npz']; json_hash=lambda name:hashlib.sha256((out/name).read_bytes()).hexdigest()
before={name:json_hash(name) for name in names[:2]}; heads={name:dict(np.load(out/name)) for name in names[2:]}
rerun=subprocess.run([sys.executable,str(out/'group_oof.py')],capture_output=True,text=True);assert rerun.returncode==0,rerun.stderr
assert before=={name:json_hash(name) for name in names[:2]}
for name,arrays in heads.items():
    with np.load(out/name) as again:
        assert all(np.array_equal(v,again[k]) for k,v in arrays.items())
rr=readj(out/'results.json'); vv=readj(out/'heldout_detector.json');
manifest={'run_from':'/home/jovyan/work','replay':['python runs/dinov2_material_stage1/'+x for x in ['extract_embeddings.py','group_oof.py','material_classifier.py']],'versions':{k:version(k) for k in ['numpy','torch','torchvision','scikit-learn','Pillow']},'verified_identical_oof':before,'verified_identical_head_arrays':True,'inputs':{},'files':{}}
for p in [data/'labels/real_train.json',root/'baseline_v0/split_v0.json',root/'datasets/score.py',old/'oof_predictions.json',root/'runs/take_probability_20261121_e4374505/train.json',root/'runs/diagnostic_v2_20261121/capture.json']:
    manifest['inputs'][str(p.relative_to(root))]=hashlib.sha256(p.read_bytes()).hexdigest()
for p in out.iterdir():
    if p.is_file() and p.name not in ['manifest.json','REPORT.md']:manifest['files'][p.name]=json_hash(p.name)
(out/'manifest.json').write_text(json.dumps(manifest,indent=2));print('Hashes and learned arrays matched on independent repeat.',before);print('Holdout', vv['metrics']); print('orientation',rr['classes']); print('per object macro', {k:np.mean([v[k] for v in rr['objects'].values()]) for k in ['det_RF','det_DINO']});print('files',list(manifest['files']))

In [ ]:
report_path=out/'REPORT.md'
report_path.write_text("# Astramis · DINOv2 ViT-S/14 · материал\n\nСтатус эксперимента — INCONCLUSIVE для замены material_simple_v1. Обученный классификатор и результаты групповой проверки сохранены. Публичная отправка не выполнялась.\n\n| Проверка на одинаковых данных | RandomForest | DINOv2 + линейная голова |\n|---|---:|---:|\n| OOF, официальный M, рамки YOLO | 0,486710 | 0,636919 |\n| OOF, accuracy, рамки YOLO | 0,478788 | 0,630303 |\n| OOF, macro-F1, рамки YOLO | 0,482526 | 0,596581 |\n| Независимый val, официальный M, рамки YOLO | 0,766106 | 0,710084 |\n| Независимый val, accuracy | 28/49 | 28/49 |\n\nНастоящие RGB-кадры из real/train. В OOF 165 аннотаций, 32 физических предмета и 49 пустых кадров. Использованы четыре исторических непересекающихся набора object_id. Разбиение детектора зафиксировано ранее. Детектор видел train-кадры при своём обучении, поэтому OOF характеризует обобщение головы при фиксированных рамках детектора. На независимом val — 49 кадров и 8 других предметов. Присутствуют только plastic (34 кадра), wood (14) и paper (1). Число верных ответов на plastic выросло с 15 до 19, на wood упало с 12 до 8. Обе модели правильно назвали единственный paper.\n\nЭнкодер Meta DINOv2 ViT-S/14 загружен строго из весов dinov2_vits14_pretrain.pth, полностью заморожен. Из RGB-кропа получен 384-мерный CLS-вектор. Предобработка согласно исходному коду Meta: bicubic Resize(256), CenterCrop(224), ImageNet mean/std. На GT-кропах обучалась только линейная multinomial logistic regression с L2. Регуляризация выбиралась по pooled macro-recall во внутренних групповых разбиениях только среди обучающих предметов внешнего фолда. Классы с единственным оставшимся представителем использовались только для внутреннего обучения. Внешние метрики получены на всех отложенных предметах.\n\nНа GT-кропах OOF accuracy 0,630303 и macro-F1 0,596184. На рамках YOLO DINOv2 выиграл по M в 3 фолдах из 4, по доле верных классификаций улучшил 16 предметов, сохранил результат 10, ухудшил 6. В OOF ошибки plastic → metal сократились с 12 до 2 и metal → plastic с 13 до 6. На glass 20/20 верных (3 предмета), на mineral 12/15 (5 предметов), на paper 3/27 (4 предмета). Для paper baseline дал 12/27. Полные матрицы ошибок, порядок столбцов, состав фолдов и метрики по предметам записаны в results.json. Результаты по кадрам — oof_rows.json и heldout_detector.json.\n\nВ head_train32.npz сохранена голова для независимой валидации. В head_all40.npz — финальная голова по 214 аннотациям всех 40 предметов. Метрики финальной головы на её обучающих данных не вычислялись. Обе головы содержат классы, матрицу весов, смещение и список fit_objects. material_classifier.MaterialClassifier.classify принимает PIL RGB-изображение и рамку xyxy. Официальный M рассчитан через datasets/score.py; остальные компоненты решения не переобучались. Общий и публичный Score кандидата не проверялись.\n\n## Воспроизведение на StackMoreLayers\n\nИз /home/jovyan/work выполнить по порядку:\n\n~~~sh\npython runs/dinov2_material_stage1/extract_embeddings.py\npython runs/dinov2_material_stage1/group_oof.py\npython runs/dinov2_material_stage1/material_classifier.py\n~~~\n\nИсходные открытые веса и код Meta указаны в encoder.json. Для использования внешней модели в соревновании правила требуют назвать источник в обсуждении. Автоматической публикации не выполнялось.\n",encoding='utf-8')
print(report_path,report_path.stat().st_size); print('Manifest present:',(out/'manifest.json').is_file())